In [1]:
# !pip install tensorflow-gpu==2.6.2

In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import polars as pl
import numpy as np
import pickle
import gc
from deepctr.feature_column import SparseFeat, VarLenSparseFeat

# from preprocess import (
#     gen_data_set,
#     gen_model_input,
#     gen_data_set_sdm,
#     gen_model_input_sdm,
# )
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from tensorflow.python.keras import backend as K
from tensorflow.python.keras.models import Model

from deepmatch.models import *
from deepmatch.utils import sampledsoftmaxloss, NegativeSampler
import sys

sys.path.append("../src/otto_recommender")
import utils
import deepmatch_preprocess

2023-01-23 22:41:22.060336: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2023-01-23 22:41:22.060371: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [2]:
import tensorflow

tensorflow.__version__

'2.8.0'

In [3]:
from tqdm.notebook import tqdm

tqdm.pandas()

In [4]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch13_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    new_split = ""


config = Config()

In [5]:
%%time

with open('train_set.pickle', 'rb') as f:
    train_set = pickle.load(f)

CPU times: user 3min 13s, sys: 41.5 s, total: 3min 55s
Wall time: 3min 55s


In [6]:
len(train_set), len(train_set[0]), len(train_set[-1])

(54085837, 9, 9)

In [7]:
SEQ_LEN = 10
SEQ_LEN_short = 3
SEQ_LEN_prefer = 10
negsample = 0

In [8]:
%%time

train_model_input, train_label = deepmatch_preprocess.gen_model_input_sdm(
    train_set, 
    seq_short_max_len=SEQ_LEN_short, 
    seq_prefer_max_len=SEQ_LEN_prefer, 
    item_pad=954978, 
    type_pad=3
)

prefer_train_seq_types: 100%|██| 54085837/54085837 [00:17<00:00, 3007740.59it/s]


CPU times: user 12min 53s, sys: 1min 17s, total: 14min 11s
Wall time: 14min 9s


In [9]:
%%time

with open('lbe_user.pickle', 'rb') as f:
    lbe_user = pickle.load(f)
    
with open('lbe_item.pickle', 'rb') as f:
    lbe_item = pickle.load(f)

CPU times: user 4.63 ms, sys: 40.1 ms, total: 44.7 ms
Wall time: 44.3 ms


In [10]:
%%time

embedding_dim = 32

user_feature_columns = [SparseFeat('user_id', 13366604 + 1, 16),
                        VarLenSparseFeat(SparseFeat('short_item_id', 954978 + 1, embedding_dim,
                                                    embedding_name="item_id"), SEQ_LEN_short, 'mean',
                                         'short_sess_length'),
                        VarLenSparseFeat(SparseFeat('prefer_item_id', 954978 + 1, embedding_dim,
                                                    embedding_name="item_id"), SEQ_LEN_prefer, 'mean',
                                         'prefer_sess_length'),
                        VarLenSparseFeat(SparseFeat('short_types', 3, embedding_dim,
                                                    embedding_name="types"), SEQ_LEN_short, 'mean',
                                         'short_sess_length'),
                        VarLenSparseFeat(SparseFeat('prefer_types', 3, embedding_dim,
                                                    embedding_name="types"), SEQ_LEN_prefer, 'mean',
                                         'prefer_sess_length'),
                        ]


item_feature_columns = [SparseFeat('item_id', 954978 + 1, embedding_dim)]

CPU times: user 239 µs, sys: 30 µs, total: 269 µs
Wall time: 275 µs


In [11]:
%%time

from collections import Counter
train_counter = Counter(train_model_input['item_id'])
item_count = [train_counter.get(i, 0) for i in range(item_feature_columns[0].vocabulary_size)]
sampler_config = NegativeSampler('frequency', num_sampled=255, item_name="item_id", item_count=item_count)

CPU times: user 15.1 s, sys: 94.9 ms, total: 15.2 s
Wall time: 15.1 s


In [12]:
%%time

# 3.Define Model and train

import tensorflow as tf
if tf.__version__ >= '2.0.0':
    tf.compat.v1.disable_eager_execution()
else:
    K.set_learning_phase(True)

model = SDM(user_feature_columns, item_feature_columns, history_feature_list=['item_id', 'types'],
            units=embedding_dim, sampler_config=sampler_config)

model.compile(optimizer="adam", loss=sampledsoftmaxloss)

history = model.fit(train_model_input, train_label,  # train_label,
                    batch_size=512, epochs=1, verbose=1, validation_split=0.0, )

2023-01-23 22:59:49.628963: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2023-01-23 22:59:49.629042: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory
2023-01-23 22:59:49.629089: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublasLt.so.11'; dlerror: libcublasLt.so.11: cannot open shared object file: No such file or directory
2023-01-23 22:59:49.630654: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcusolver.so.11'; dlerror: libcusolver.so.11: cannot open shared object file: No such file or directory
2023-01-23 22:59:49.630706: W tensorflow/stream_executor/platform/default/dso_loader

Instructions for updating:
Please use `keras.layers.RNN(cell)`, which is equivalent to this API
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Train on 54085837 samples


2023-01-23 23:00:48.310100: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


FailedPreconditionError: Could not find variable rnn/dynamic_multi_rnn/multi_rnn_cell/cell_0/basic_lstm_cell/kernel. This could mean that the variable has been deleted. In TF1, it can also mean the variable is uninitialized. Debug info: container=localhost, status error message=Resource localhost/rnn/dynamic_multi_rnn/multi_rnn_cell/cell_0/basic_lstm_cell/kernel/N10tensorflow3VarE does not exist.
	 [[{{node rnn/dynamic_multi_rnn/multi_rnn_cell/cell_0/basic_lstm_cell/kernel/Read/ReadVariableOp}}]]

In [10]:
train_set[-2]

(13366602, 545070, 1, [], [], 0, 0, [], [])

In [11]:
dict_types = dict()
dict_types["clicks"] = 0
dict_types["carts"] = 1
dict_types["orders"] = 2

type_labels = pl.DataFrame(
    {"type": list(dict_types.keys()), "values": list(dict_types.values())}
)

In [5]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config, new_split=False)
# X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
#     config, 
#     X_train, 
#     X_val, 
#     X_val_seq, 
#     ordinal_encoder, 
#     return_only_val=True, 
#     truncate_val=True
# )
pad = ordinal_encoder.unknown_value

# X_train = X_train.drop("count")
# X_train = X_train.drop("n_session_right")
# X_val = X_val.drop("count")
# X_val_seq = X_val_seq.drop("count")

CPU times: user 15min 4s, sys: 1min 52s, total: 16min 56s
Wall time: 3min 26s


In [6]:
%%time

X_train = (
    X_train.join(type_labels, on=config.col_type, how="left")
    .drop(columns=[config.col_type])
    .rename({"values": config.col_type})
)

CPU times: user 15.6 s, sys: 14.7 s, total: 30.4 s
Wall time: 2.92 s


In [7]:
# %%time

# X_train = (
#     X_train.groupby([config.col_user])
#     .agg(
#         [
#             pl.col(config.col_aid).list(),
#             pl.col(config.col_type).list(),

#         ]
#     ).filter(pl.col(config.col_aid).apply(lambda x: len(x)) >= 2)
# )

In [8]:
# %%time

# X_train = X_train.to_pandas()
# X_train[config.col_train_seq] = X_train[config.col_aid].progress_apply(lambda x: x[:-1])
# X_train[config.col_target] = X_train[config.col_aid].progress_apply(lambda x: x[-1])

# X_train['train_seq_type'] = X_train[config.col_type].progress_apply(lambda x: x[:-1])
# X_train['target_type'] = X_train[config.col_type].progress_apply(lambda x: x[-1])
# X_train = pl.from_pandas(X_train)

In [9]:
X_train["aid"].max()

954978

In [10]:
X_train["user"].max()

14571534

In [11]:
%%time

X_train = X_train.with_columns(
        [
            pl.col(config.col_aid)
            .cast(pl.Int32, strict=False)
            .alias(config.col_aid),
            pl.col(config.col_user)
            .cast(pl.Int32, strict=False)
            .alias(config.col_user),
            pl.col(config.col_type)
            .cast(pl.Int8, strict=False)
            .alias(config.col_type),
        ]
    )

CPU times: user 2.05 s, sys: 1.69 s, total: 3.74 s
Wall time: 1.49 s


In [12]:
X_train

user,aid,ts,n_session,num_action,type
i32,i32,i64,i32,i32,i8
14250687,467903,1662328738422,2,4,0
14250687,467903,1662328710275,2,5,0
14571534,139680,1662328709214,0,4,0
14571149,278804,1662328707619,0,4,0
14250687,467903,1662328701548,2,6,0
14565076,92435,1662328700064,0,4,0
14565076,77507,1662328695403,0,5,0
13029210,695258,1662328695309,1,4,0
14571519,545070,1662328691389,0,4,0


In [13]:
len(X_train[config.col_user].unique())

13366604

In [14]:
X_train[config.col_user].max()

14571534

In [15]:
X_train = X_train.to_pandas()

In [16]:
%%time

SEQ_LEN = 10
SEQ_LEN_short = 3
SEQ_LEN_prefer = 10
negsample = 0

# 1.Label Encoding for sparse features, and process sequence features with `gen_date_set` and `gen_model_input`
dict_lbes = dict()

feature_max_idx = dict()
feature_max_idx[config.col_type] = 2
feature_max_idx[config.col_aid] = pad
for feature in [config.col_user]:
    unknown_value = len(X_train[feature].unique())
    lbe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=unknown_value)
    X_train[feature] = lbe.fit_transform(X_train[feature].values.reshape(-1, 1)).astype('int32')
    feature_max_idx[feature] = unknown_value
    dict_lbes[feature] = lbe

CPU times: user 2min 40s, sys: 2.04 s, total: 2min 42s
Wall time: 2min 42s


In [17]:
X_train.head()

,user,aid,ts,n_session,num_action,type
0,13281307,467903,1662328738422,2,4,0
1,13281307,467903,1662328710275,2,5,0
2,13366603,139680,1662328709214,0,4,0
3,13366560,278804,1662328707619,0,4,0
4,13281307,467903,1662328701548,2,6,0


In [18]:
X_train["type"]

0            0
1            0
2            0
3            0
4            0
            ..
220423446    0
220423447    1
220423448    0
220423449    0
220423450    1
Name: type, Length: 220423451, dtype: int8

In [19]:
col_ts = config.col_ts
col_item = config.col_aid
col_type = config.col_type
col_user = config.col_user

In [20]:
%%time

train_set = deepmatch_preprocess.gen_data_set_sdm(X_train, seq_short_max_len=2, seq_prefer_max_len=10)

100%|█████████████████████████████| 13366604/13366604 [38:46<00:00, 5746.33it/s]

9
CPU times: user 39min 23s, sys: 1min 42s, total: 41min 6s
Wall time: 40min 52s


In [26]:
del X_train, X_val, X_val_seq
gc.collect()

In [25]:
%%time

import pickle

with open('train_set.pickle', 'wb') as f:
    pickle.dump(train_set, f, protocol=4)

CPU times: user 4min 49s, sys: 48 s, total: 5min 37s
Wall time: 5min 38s


In [ ]:
stop

In [ ]:
data[data["user"] == 0].head(15)

In [ ]:
    # train_uid = np.array([line[0] for line in train_set])
    # train_iid = np.array([line[1] for line in train_set])
    # train_label = np.array([line[2] for line in train_set])
    # short_train_seq = [line[3] for line in train_set]
    # prefer_train_seq = [line[4] for line in train_set]
    # train_short_len = np.array([line[5] for line in train_set])
    # train_prefer_len = np.array([line[6] for line in train_set])
    # short_train_seq_genres = np.array([line[7] for line in train_set])
    # prefer_train_seq_genres = np.array([line[8] for line in train_set])

In [ ]:
train_set[13]

In [ ]:
item_id_types_map[954978]

In [ ]:
%%time

sparse_features = [
    "user",
    "aid",
    "type",
]

SEQ_LEN = 10
negsample = 0

feature_max_idx = dict()
feature_lbe = dict()
for feature in sparse_features:
    lbe = LabelEncoder()
    data[feature] = lbe.fit_transform(data[feature]) + 1
    feature_max_idx[feature] = data[feature].max() + 1
    feature_lbe[feature] = lbe

In [ ]:
feature_max_idx

In [ ]:
%%time

user_profile = data[["user", "aid", "type"]].drop_duplicates("user")

item_profile = data[["movie_id", "genres"]].drop_duplicates("movie_id")

user_profile.set_index("user_id", inplace=True)

user_item_list = data.groupby("user_id")["movie_id"].apply(list)

train_set, test_set = gen_data_set(data, SEQ_LEN, negsample)

train_model_input, train_label = gen_model_input(train_set, user_profile, SEQ_LEN)
test_model_input, test_label = gen_model_input(test_set, user_profile, SEQ_LEN)

In [ ]:
%%time

embedding_dim = 32

user_feature_columns = [SparseFeat('user_id', feature_max_idx['user_id'], 16),
                        SparseFeat("gender", feature_max_idx['gender'], 16),
                        SparseFeat("age", feature_max_idx['age'], 16),
                        SparseFeat("occupation", feature_max_idx['occupation'], 16),
                        SparseFeat("zip", feature_max_idx['zip'], 16),
                        VarLenSparseFeat(SparseFeat('hist_movie_id', feature_max_idx['movie_id'], embedding_dim,
                                                    embedding_name="movie_id"), SEQ_LEN, 'mean', 'hist_len'),
                        VarLenSparseFeat(SparseFeat('hist_genres', feature_max_idx['genres'], embedding_dim,
                                embedding_name="genres"), SEQ_LEN, 'mean', 'hist_len'),
                        ]

item_feature_columns = [SparseFeat('movie_id', feature_max_idx['movie_id'], embedding_dim),
                        SparseFeat('genres', feature_max_idx['genres'], embedding_dim)
                       ]

from collections import Counter
train_counter = Counter(train_model_input['movie_id'])
item_count = [train_counter.get(i,0) for i in range(item_feature_columns[0].vocabulary_size)]
sampler_config = NegativeSampler('inbatch', num_sampled=255, item_name="movie_id", item_count=item_count)

In [ ]:
%%time

import tensorflow as tf
if tf.__version__ >= '2.0.0':
    tf.compat.v1.disable_eager_execution()
else:
    K.set_learning_phase(True)
    
model = DSSM(user_feature_columns, item_feature_columns,user_dnn_hidden_units=(128,64, embedding_dim),
             item_dnn_hidden_units=(64, embedding_dim,),loss_type='softmax',sampler_config=sampler_config)

model.compile(optimizer="adam", loss=sampledsoftmaxloss)

history = model.fit(train_model_input, train_label,  # train_label,
                    batch_size=256, epochs=3, verbose=1, validation_split=0.0, )

# 4. Generate user features for testing and full item features for retrieval
test_user_model_input = test_model_input
all_item_model_input = {"movie_id": item_profile['movie_id'].values,"genres": item_profile['genres'].values}

user_embedding_model = Model(inputs=model.user_input, outputs=model.user_embedding)
item_embedding_model = Model(inputs=model.item_input, outputs=model.item_embedding)

user_embs = user_embedding_model.predict(test_user_model_input, batch_size=2 ** 12)
# user_embs = user_embs[:, i, :]  # i in [0,k_max) if MIND
item_embs = item_embedding_model.predict(all_item_model_input, batch_size=2 ** 12)

print(user_embs.shape)
print(item_embs.shape)